# 🏥 MedicalVLM — Google Colab Setup
### LLaVA-1.5-7b-hf | FastAPI | ngrok

**Steps:**
1. Run all pip installs
2. Set your secrets (Supabase keys + ngrok token)
3. Clone / mount your project
4. Start the backend via uvicorn + ngrok
5. Test with a sample image

> ⚡ Runtime → Change runtime type → **T4 GPU** (recommended)

In [ ]:
# ── Cell 1: Install all dependencies ─────────────────────────────────
!pip install -q \
  torch torchvision --index-url https://download.pytorch.org/whl/cu118 \
  transformers>=4.37.0 \
  accelerate>=0.26.0 \
  bitsandbytes>=0.42.0 \
  fastapi uvicorn[standard] python-multipart \
  supabase python-dotenv \
  fpdf2 opencv-python-headless Pillow numpy \
  pytorch-grad-cam \
  pyngrok \
  requests nest_asyncio

print('✅ All packages installed')

In [ ]:
# ── Cell 2: Verify CUDA ───────────────────────────────────────────────
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 3: Set your secrets ─────────────────────────────────────────
# Option A: Use Colab Secrets panel (left sidebar 🔑) — recommended
# Option B: Set them directly here (NOT recommended for shared notebooks)

import os

# Try Colab secrets first, then fall back to hardcoded strings
try:
    from google.colab import userdata
    os.environ['SUPABASE_URL']         = userdata.get('SUPABASE_URL')
    os.environ['SUPABASE_ANON_KEY']    = userdata.get('SUPABASE_ANON_KEY')
    os.environ['SUPABASE_SERVICE_KEY'] = userdata.get('SUPABASE_SERVICE_KEY')
    NGROK_TOKEN                        = userdata.get('NGROK_TOKEN')
    print('✅ Loaded secrets from Colab Secrets')
except Exception:
    # ↓↓ Replace these with your actual values if not using Colab Secrets
    os.environ['SUPABASE_URL']         = 'https://YOUR_PROJECT.supabase.co'
    os.environ['SUPABASE_ANON_KEY']    = 'your-anon-key'
    os.environ['SUPABASE_SERVICE_KEY'] = 'your-service-role-key'
    NGROK_TOKEN                        = 'your-ngrok-authtoken'
    print('⚠️  Using hardcoded secrets — add to Colab Secrets for better security')

os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['PORT']    = '8000'

In [ ]:
# ── Cell 4: Mount Google Drive OR clone repo ──────────────────────────
import os

USE_DRIVE = False   # ← set True if your project is in Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/MedicalVLM'  # ← adjust path
else:
    # Clone from GitHub (replace with your repo URL)
    REPO_URL = 'https://github.com/YOUR_USERNAME/MedicalVLM.git'
    if not os.path.exists('/content/MedicalVLM'):
        !git clone {REPO_URL} /content/MedicalVLM
    PROJECT_DIR = '/content/MedicalVLM'

os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

In [ ]:
# ── Cell 5: Write .env for the backend ───────────────────────────────
import os

env_content = f"""SUPABASE_URL={os.environ['SUPABASE_URL']}
SUPABASE_ANON_KEY={os.environ['SUPABASE_ANON_KEY']}
SUPABASE_SERVICE_KEY={os.environ['SUPABASE_SERVICE_KEY']}
HF_HOME=/content/hf_cache
PORT=8000
"""

with open('.env', 'w') as f:
    f.write(env_content)

print('✅ .env file written')

In [ ]:
# ── Cell 6: Configure ngrok ───────────────────────────────────────────
from pyngrok import ngrok, conf

ngrok.set_auth_token(NGROK_TOKEN)
print('✅ ngrok authenticated')

In [ ]:
# ── Cell 7: Start FastAPI via uvicorn in background + open ngrok tunnel
import subprocess, time, os, nest_asyncio
from pyngrok import ngrok

nest_asyncio.apply()

# Kill any previous process on port 8000
!fuser -k 8000/tcp 2>/dev/null || true
time.sleep(1)

# Start uvicorn in background
server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'backend.api.main:app',
     '--host', '0.0.0.0', '--port', '8000', '--log-level', 'info'],
    cwd=os.getcwd(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print('Server starting… waiting 30 s for model to load…')
time.sleep(30)  # LLaVA takes ~20-40 s to load on Colab T4

# Open ngrok tunnel
tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url
print(f'\n✅ Backend live at: {PUBLIC_URL}')
print(f'   Health check  : {PUBLIC_URL}/health')
print(f'\n👉 Set VITE_API_URL={PUBLIC_URL} in your frontend .env')

In [ ]:
# ── Cell 8: Health check ─────────────────────────────────────────────
import requests, json

resp = requests.get(f'{PUBLIC_URL}/health')
print(json.dumps(resp.json(), indent=2))

In [ ]:
# ── Cell 9: Test inference with a sample image ────────────────────────
# You need a valid Supabase JWT — get one by logging in via the frontend
# or by calling supabase.auth.sign_in_with_password() directly

import requests
from supabase import create_client
import os

SUPABASE_TEST_EMAIL    = 'your@email.com'   # ← replace
SUPABASE_TEST_PASSWORD = 'yourpassword'      # ← replace

sb       = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_ANON_KEY'])
auth_res = sb.auth.sign_in_with_password({'email': SUPABASE_TEST_EMAIL, 'password': SUPABASE_TEST_PASSWORD})
JWT      = auth_res.session.access_token
print('✅ Got JWT')

# Download a sample public chest X-ray
SAMPLE_URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/c/c8/Chest_Xray_PA_3-8-2010.png/800px-Chest_Xray_PA_3-8-2010.png'
img_bytes  = requests.get(SAMPLE_URL).content

# Call /generate
resp = requests.post(
    f'{PUBLIC_URL}/generate',
    headers={'Authorization': f'Bearer {JWT}'},
    files={'file': ('sample.png', img_bytes, 'image/png')},
    data={'patient_history': '60-year-old female with persistent cough'},
    timeout=300,
)

if resp.status_code == 200:
    data = resp.json()
    print('\n=== REPORT ===')
    print(data['report'])
    print(f'\nProcessing time: {data["processing_time_ms"]} ms')
    print(f'Heatmap present: {bool(data["heatmap_base64"])}')
else:
    print(f'Error {resp.status_code}: {resp.text}')

In [ ]:
# ── Cell 10: Test /briefing endpoint ─────────────────────────────────
resp = requests.post(
    f'{PUBLIC_URL}/briefing',
    headers={'Authorization': f'Bearer {JWT}'},
    files={'file': ('sample.png', img_bytes, 'image/png')},
    timeout=180,
)

if resp.status_code == 200:
    print('\n=== PATIENT BRIEFING ===')
    print(resp.json()['briefing'])
else:
    print(f'Error {resp.status_code}: {resp.text}')

In [ ]:
# ── Cell 11: Test /export-pdf endpoint ───────────────────────────────
resp = requests.post(
    f'{PUBLIC_URL}/export-pdf',
    headers={'Authorization': f'Bearer {JWT}'},
    files={'file': ('sample.png', img_bytes, 'image/png')},
    data={'patient_history': '60-year-old female with persistent cough'},
    timeout=300,
)

if resp.status_code == 200:
    with open('/content/test_report.pdf', 'wb') as f:
        f.write(resp.content)
    print('✅ PDF saved to /content/test_report.pdf')
    from google.colab import files
    files.download('/content/test_report.pdf')
else:
    print(f'Error {resp.status_code}: {resp.text}')

In [ ]:
# ── Cell 12: View server logs (last 50 lines) ─────────────────────────
import subprocess
out, _ = server_proc.communicate(timeout=0) if server_proc.poll() is not None else (b'', b'')
if out:
    print(out.decode()[-3000:])
else:
    print('Server still running — stdout not available in Popen without communicate()')
    # Use: !ps aux | grep uvicorn   to verify

In [ ]:
# ── Cell 13: Stop server & close ngrok tunnel ─────────────────────────
from pyngrok import ngrok
ngrok.kill()
server_proc.terminate()
print('✅ Server stopped')